# Laboratorium 03 — backpropagation od podstaw w NumPy

Materiał towarzyszący [`README.md`](README.md). Implementujesz forward i
backward dla sieci `Linear(1, hidden) -> tanh -> Linear(hidden, 1)`
wyłącznie w NumPy, weryfikujesz gradienty względem PyTorch `autograd`, i
trenujesz regresję nieliniową własnymi gradientami.




In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch  # noqa: F401 - uzywany w gradient_check (sekcja 4)

rng = np.random.default_rng(42)


## 1. Dane: regresja nieliniowa (gotowe)

$y = \sin(2x) + 0{,}3x^2 + \varepsilon$ — funkcja wyraźnie nieliniowa.


In [ ]:
N = 150
X = rng.uniform(-3, 3, size=(N, 1))
noise = rng.normal(0, 0.2, size=(N, 1))
Y = np.sin(2 * X) + 0.3 * X**2 + noise

plt.figure(figsize=(5, 4))
plt.scatter(X, Y, s=15, alpha=0.6)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dane: regresja nieliniowa")
plt.tight_layout()
plt.show()


## 2. Forward pass — Twoje zadanie

`forward(X, W1, b1, W2, b2)` dla `Linear(1, hidden) -> tanh -> Linear(hidden, 1)`.
Zwróć wszystkie wartości pośrednie: `Z1, A1, Z2`.


In [ ]:
def forward(X, W1, b1, W2, b2):
    """Zwraca (Z1, A1, Z2). X: (N,1), W1: (hidden,1), W2: (1,hidden)."""
    # TODO: Z1 = X @ W1.T + b1 ; A1 = tanh(Z1) ; Z2 = A1 @ W2.T + b2
    raise NotImplementedError("TODO: forward")


## 3. Backward pass — Twoje zadanie

To jest rdzeń laboratorium: `backward(X, Y, Z1, A1, Z2, W2)` zwraca
`(dW1, db1, dW2, db2)`. Użyj wzorów z wykładu (sekcja *Wersja wsadowa*),
podstawiając pochodną **tanh** ($1-\tanh^2(z)$) zamiast sigmoid.


In [ ]:
def backward(X, Y, Z1, A1, Z2, W2):
    """Zwraca (dW1, db1, dW2, db2), ksztalty jak (W1, b1, W2, b2)."""
    # TODO:
    # 1. Delta2 = dL/dZ2 (MSE, uwzglednij uśrednianie po batchu: wzor z wykladu)
    # 2. dW2 = Delta2.T @ A1 ; db2 = suma Delta2 po batchu
    # 3. dA1 = Delta2 @ W2 ; Delta1 = dA1 * tanh'(Z1)  [tanh'(z) = 1 - tanh(z)**2]
    # 4. dW1 = Delta1.T @ X ; db1 = suma Delta1 po batchu
    raise NotImplementedError("TODO: backward")


## 4. Weryfikacja gradientu — Twoje zadanie

Zbuduj ten sam forward pass w PyTorch z `requires_grad=True`, wywołaj
`backward()`, porównaj z gradientami ze swojej funkcji `backward`.


In [ ]:
def gradient_check(X, Y, W1, b1, W2, b2):
    """Zwraca dict {"dW1","db1","dW2","db2"} -> max|numpy - torch|."""
    # TODO:
    # 1. policz wlasne gradienty: forward(...) potem backward(...)
    # 2. zbuduj rownowazny forward w torch z W1,b1,W2,b2 jako
    #    torch.tensor(..., requires_grad=True); torch.tensor(X), torch.tensor(Y)
    # 3. policz strate MSE w torch, wywolaj .backward()
    # 4. zwroc maksymalna roznice bezwzgledna dla kazdego z 4 gradientow
    raise NotImplementedError("TODO: gradient_check")


In [ ]:
hidden_size = 16
W1_init = rng.normal(0, 0.5, size=(hidden_size, 1))
b1_init = np.zeros(hidden_size)
W2_init = rng.normal(0, 0.5, size=(1, hidden_size))
b2_init = np.zeros(1)

diffs = gradient_check(X, Y, W1_init, b1_init, W2_init, b2_init)
for name, diff in diffs.items():
    print(f"{name}: max|numpy - torch| = {diff:.2e}")
assert max(diffs.values()) < 1e-8, (
    "gradient_check nie przeszedl - sprawdz backward() przed treningiem"
)
print("\nOK: gradienty zgadzaja sie z PyTorch autograd.")


## 5. Trening — Twoje zadanie

`train(X, Y, hidden_size, lr, epochs)`: inicjalizuje wagi losowo, trenuje
własnym `forward`/`backward`, aktualizuje parametry ręcznie (bez
`torch.optim`), zwraca `(W1, b1, W2, b2, loss_history)`.


In [ ]:
def train(X, Y, hidden_size, lr, epochs, seed=0):
    """Zwraca (W1, b1, W2, b2, loss_history)."""
    # TODO:
    # 1. zainicjuj W1 (hidden,1), b1 (hidden,), W2 (1,hidden), b2 (1,) losowo
    #    (np. rng.normal(0, 0.5, size=...), rng = np.random.default_rng(seed))
    # 2. w petli po epochs: forward -> strata MSE -> backward -> aktualizacja
    #    W -= lr * dW (analogicznie dla b) -> zbieraj strate do listy
    raise NotImplementedError("TODO: train")


In [ ]:
W1_fit, b1_fit, W2_fit, b2_fit, loss_history = train(X, Y, hidden_size=16, lr=0.05, epochs=3000)

baseline_loss = np.mean((Y - Y.mean()) ** 2)
final_loss = loss_history[-1]
print(f"Baseline (srednia) strata: {baseline_loss:.4f}")
print(f"Wytrenowana siec, strata koncowa: {final_loss:.4f}")


## 6. Interpretacja — Twoje zadanie


In [ ]:
x_line = np.linspace(-3, 3, 200).reshape(-1, 1)
_, _, y_line = forward(x_line, W1_fit, b1_fit, W2_fit, b2_fit)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(loss_history)
axes[0].set_xlabel("epoka")
axes[0].set_ylabel("strata (MSE)")
axes[0].set_title("Krzywa straty")

axes[1].scatter(X, Y, s=15, alpha=0.5, label="dane")
axes[1].plot(x_line, y_line, color="tab:red", label="model")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Dopasowanie")
axes[1].legend()

plt.tight_layout()
plt.show()


**Zadanie (3–5 zdań, komórka Markdown):** czy krzywa straty wygląda tak,
jak oczekujesz po wykładzie 1 i 3 (kształt, tempo spadku)? Czy dopasowanie
dobrze oddaje kształt funkcji w całym przedziale $x$, czy widzisz region z
systematycznym (nie losowym) odchyleniem od danych — a jeśli nie, czy
pozostały rozrzut punktów wokół krzywej modelu da się wytłumaczyć samym
szumem, z jakim dane zostały wygenerowane (sekcja 1)?

_(Twoja odpowiedź tutaj.)_


## 7. Błędny gradient — Twoje zadanie

Druga wersja `backward`, z celowym błędem, trenująca identyczną sieć na
tych samych danych.


In [ ]:
def backward_buggy(X, Y, Z1, A1, Z2, W2):
    N = X.shape[0]
    Delta2 = (2.0 / N) * (Z2 - Y)
    dW2 = Delta2.T @ A1
    db2 = Delta2.sum(axis=0)

    dA1 = Delta2 @ W2
    Delta1 = dA1  # <- błąd jest tutaj
    dW1 = Delta1.T @ X
    db1 = Delta1.sum(axis=0)
    return dW1, db1, dW2, db2


def train_with_backward_fn(X, Y, hidden_size, lr, epochs, backward_fn, seed=0):
    g = np.random.default_rng(seed)
    W1 = g.normal(0, 0.5, size=(hidden_size, 1))
    b1 = np.zeros(hidden_size)
    W2 = g.normal(0, 0.5, size=(1, hidden_size))
    b2 = np.zeros(1)

    loss_history = []
    for _ in range(epochs):
        Z1, A1, Z2 = forward(X, W1, b1, W2, b2)
        loss_history.append(np.mean((Z2 - Y) ** 2))
        dW1, db1, dW2, db2 = backward_fn(X, Y, Z1, A1, Z2, W2)
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2
    return W1, b1, W2, b2, loss_history


_, _, _, _, loss_history_correct = train_with_backward_fn(
    X, Y, hidden_size=16, lr=0.05, epochs=3000, backward_fn=backward
)
_, _, _, _, loss_history_buggy = train_with_backward_fn(
    X, Y, hidden_size=16, lr=0.05, epochs=3000, backward_fn=backward_buggy
)

plt.figure(figsize=(5.5, 4))
plt.plot(loss_history_correct, label="backward (poprawny)")
plt.plot(loss_history_buggy, label="backward_buggy")
plt.axhline(baseline_loss, color="black", linestyle="--", linewidth=0.8, label="baseline (średnia)")
plt.xlabel("epoka")
plt.ylabel("strata (MSE)")
plt.title("Wpływ błędnego gradientu na trening")
plt.legend()
plt.tight_layout()
plt.show()

print(f"poprawny backward, strata koncowa: {loss_history_correct[-1]:.4f}")
print(f"backward_buggy,    strata koncowa: {loss_history_buggy[-1]:.4f}")
print(f"baseline (srednia):                {baseline_loss:.4f}")


**Zadanie (3–5 zdań, komórka Markdown):** zidentyfikuj, czego brakuje w
`backward_buggy` względem `backward` (odwołaj się do konkretnego wzoru z
wykładu — które mnożenie zostało pominięte) i wyjaśnij, dlaczego akurat
ten brak prowadzi do zaobserwowanego przebiegu krzywej straty.

_(Twoja odpowiedź tutaj.)_
